# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

I made a 90 days time window selected from latest 3 months from date 2026-04-01 to 2026-06-30.<br><br>
One row = One client's one reported content page needed refresh or not.<br><br> Trained on 30day window train, validated on 30day window validation and tested on 30day window test, and output label: "refresh_needed" derived by comparison between features of the 30day train window and 30day validation window.

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


<h1>Time window</h1>

In [2]:
date_range = con.sql(f"""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date,
        MAX(report_date) - MIN(report_date) as duration
    FROM {TABLES['fact_daily']}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│ start_date │  end_date  │ duration │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │      519 │
└────────────┴────────────┴──────────┘



In [3]:
clients_last_3m = con.sql(f"""
    SELECT report_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-06-30'
""")

In [4]:
print(f"Rows in last 3 months: {len(clients_last_3m):,}")

Rows in last 3 months: 33,806,178


In [5]:
# Verify the range in the new subset for 90 day window
print(" 90 day desired window")
con.sql("""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM clients_last_3m
""").show()

 90 day desired window


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│ start_date │  end_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-04-01 │ 2026-06-30 │
└────────────┴────────────┘



In [6]:
con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-06-30'
    LIMIT 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

<h1>Features:</h1>
'client_has_gsc', 'client_has_ga4', 'gsc_data_available',
       'ga4_data_available', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
       'sessions_organic', 'sessions_direct', 'sessions_referral',
       'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt',
       'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta',
       'ai_other', 'scroll_events', 'content_type', 'search_volume', 'competition',
       'competition_level', 'cpc', 'main_intent', 'backlinks', 'provider_used',
       'model_used', 'char_count', 'word_count'

<h1>Label: "refresh_needed = 1/0"</h1>
Derived label from train/validation window data from the data table columns.  

<h1>Context:</h1>
client_hash_id, content_hash_id, report_date,(later use in text data)<br> content_created_date, content_updated_date, last_optimized_date, (for derived features in future)<br> is_published, is_deleted (for data filtering)

<h1>Excluded:</h1>
1. month <br>
Reason: report_date already shows month so no need<br>
2. optimization_eligible_date<br>
Reason: leakage for output refresh_needed so dropped<br>
3. keyword_hash_id<br>
Reason: not useful for output label prediction<br>
4. url_hash_id<br>
Reason: not useful for output label prediction<br>
5. keyword_char_count<br>
Reason: not useful for output label prediction<br>
6. keyword_token_count<br>
Reason: not useful for output label prediction<br>
7. url_char_count<br>
Reason: not useful for output label prediction<br>
8. keyword_created_date<br>
Reason: not useful for output label prediction<br>
9. category_count<br>
Reason: not useful for output label prediction<br>

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_analysis_data = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE month =='2026-04'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
unit_analysis_data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [9]:
dim_c= con.sql(f"""
    SELECT *
    FROM {TABLES['dim_content']}
""").df()

In [10]:
dim_c.columns

Index(['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id',
       'keyword_char_count', 'keyword_token_count', 'url_char_count',
       'content_created_date', 'content_updated_date', 'content_type',
       'search_volume', 'competition', 'competition_level', 'cpc',
       'main_intent', 'backlinks', 'category_count', 'keyword_created_date',
       'provider_used', 'model_used', 'char_count', 'word_count',
       'last_optimized_date', 'optimization_eligible_date', 'is_published',
       'is_deleted'],
      dtype='object')

In [11]:
joined_data = con.sql(f"""
    SELECT
        f.*,
        d.keyword_char_count,
        d.keyword_token_count,
        d.content_created_date,
        d.content_updated_date,
        d.content_type,
        d.search_volume,
        d.competition,
        d.competition_level,
        d.cpc,
        d.main_intent,
        d.backlinks,
        d.category_count,
        d.keyword_created_date,
        d.provider_used,
        d.model_used,
        d.char_count,
        d.word_count,
        d.last_optimized_date,
        d.optimization_eligible_date,
        d.is_published,
        d.is_deleted
    FROM {TABLES['fact_daily']} f
    LEFT JOIN {TABLES['dim_content']} d
      ON f.client_hash_id = d.client_hash_id
      AND f.content_hash_id = d.content_hash_id
    WHERE f.month == '2026-04'
    LIMIT 2
""").df()

display(joined_data)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,2026-04-27,client_06d356715a8ff3b6,content_0059a4d4195810c9,True,True,True,False,132,0,959,...,0,2026-04-06,gemini-generate-content,gemini-3-flash-preview,13633,2087,2026-06-23,2026-08-07,True,False
1,2026-04-30,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,True,True,True,True,70,0,612,...,0,2026-04-06,gemini-generate-content,gemini-3-flash-preview,17130,2567,2026-06-23,2026-08-07,True,False


In [12]:
joined_data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month', 'keyword_char_count',
       'keyword_token_count', 'content_created_date', 'content_updated_date',
       'content_type', 'search_volume', 'competition', 'competition_level',
       'cpc', 'main_intent', 'backlinks', 'category_count',
       'keyword_created_date', 'provider_used', 'model_used', 'char_count',
       'word_count', 'last_optimized_date', 'optimization_eligible_date',
       'is_published', '

In [13]:
joined_data = joined_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id','month','keyword_char_count',
                                        'keyword_token_count', 'category_count','keyword_created_date','last_optimized_date', 'optimization_eligible_date' ])

In [14]:
joined_data.columns

Index(['client_has_gsc', 'client_has_ga4', 'gsc_data_available',
       'ga4_data_available', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
       'sessions_organic', 'sessions_direct', 'sessions_referral',
       'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt',
       'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta',
       'ai_other', 'scroll_events', 'content_created_date',
       'content_updated_date', 'content_type', 'search_volume', 'competition',
       'competition_level', 'cpc', 'main_intent', 'backlinks', 'provider_used',
       'model_used', 'char_count', 'word_count', 'is_published', 'is_deleted'],
      dtype='object')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain check

---



In [4]:
subset = unit_analysis_data[unit_analysis_data["month"] == "2026-04"]
dupes = subset.duplicated(subset=["content_hash_id", "report_date"]).sum()
print(f"Duplicate (content_hash_id, report_date) pairs: {dupes}")
print(f"Total rows this month: {len(subset)}")

Duplicate (content_hash_id, report_date) pairs: 0
Total rows this month: 10424730


Counts

In [5]:
print(f"Row count: {len(subset)}")
print(f"Date span: {subset['report_date'].min()} to {subset['report_date'].max()}")
print(f"Unique content_hash_id: {subset['content_hash_id'].nunique()}")

Row count: 10424730
Date span: 2026-04-01 00:00:00 to 2026-04-30 00:00:00
Unique content_hash_id: 362172


In [6]:
total = len(subset)
available = subset[
    (subset["gsc_data_available"] == True) &
    (subset["ga4_data_available"] == True)
]
print(f"Total rows: {total}")
print(f"Rows with both GSC and GA4 available: {len(available)}")
print(f"Rows dropped due to missing availability: {total - len(available)}")

Total rows: 10424730
Rows with both GSC and GA4 available: 462315
Rows dropped due to missing availability: 9962415


Baseline model: LogisticRegression

In [7]:
unit_analysis_data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [8]:
subset = unit_analysis_data
subset["report_date"] = pd.to_datetime(subset["report_date"])

mid_point = subset["report_date"].min() + (subset["report_date"].max() - subset["report_date"].min()) / 2
early = subset[subset["report_date"] < mid_point]   # "prev" half
late = subset[subset["report_date"] >= mid_point]   # "last" half

In [9]:
print(early.info())

<class 'pandas.core.frame.DataFrame'>
Index: 5091201 entries, 0 to 6001002
Data columns (total 31 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   report_date               datetime64[us]
 1   client_hash_id            object        
 2   content_hash_id           object        
 3   client_has_gsc            bool          
 4   client_has_ga4            bool          
 5   gsc_data_available        bool          
 6   ga4_data_available        boolean       
 7   gsc_impressions           int64         
 8   gsc_clicks                int64         
 9   gsc_sum_position          int64         
 10  gsc_avg_position          float64       
 11  ga4_pageviews             Int64         
 12  ga4_sessions              Int64         
 13  ga4_users                 Int64         
 14  ga4_engaged_sessions      Int64         
 15  ga4_total_engagement_sec  Int64         
 16  sessions_organic          Int64         
 17  sessions_dire

In [10]:
early_agg = early.groupby("content_hash_id").agg(
    gsc_impressions=("gsc_impressions", "sum"),
    sessions_organic=("sessions_organic", "sum"),

).reset_index()

late_agg = late.groupby("content_hash_id").agg(
    gsc_impressions=("gsc_impressions", "sum"),
    sessions_organic=("sessions_organic", "sum"),
).reset_index()

In [11]:
print(early_agg.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347661 entries, 0 to 347660
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   content_hash_id   347661 non-null  object
 1   gsc_impressions   347661 non-null  int64 
 2   sessions_organic  347661 non-null  Int64 
dtypes: Int64(1), int64(1), object(1)
memory usage: 8.3+ MB
None


In [12]:
outcome = early_agg.merge(late_agg, on="content_hash_id", suffixes=("_early", "_late"))
outcome["refresh_needed"] = (
    (outcome["gsc_impressions_late"] < outcome["gsc_impressions_early"] * 0.8) |
    (outcome["sessions_organic_late"] < outcome["sessions_organic_early"] * 0.8)
).astype(int)

In [24]:
print(outcome.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347661 entries, 0 to 347660
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   content_hash_id         347661 non-null  object
 1   gsc_impressions_early   347661 non-null  int64 
 2   sessions_organic_early  347661 non-null  Int64 
 3   gsc_impressions_late    347661 non-null  int64 
 4   sessions_organic_late   347661 non-null  Int64 
 5   refresh_needed          347661 non-null  int64 
dtypes: Int64(2), int64(3), object(1)
memory usage: 16.6+ MB
None


In [25]:
# checking
print("Raw rows in early:", len(early))
print("Unique content_hash_id in early:", early["content_hash_id"].nunique())
print("Rows in early_agg:", len(early_agg))

# These two numbers should match exactly:
print("Match check:", early["content_hash_id"].nunique() == len(early_agg))

Raw rows in early: 5091201
Unique content_hash_id in early: 347661
Rows in early_agg: 347661
Match check: True


In [13]:
features = early.groupby("content_hash_id").agg(
    ctr=("gsc_clicks", "sum"),                    # will divide below
    gsc_impressions_feat=("gsc_impressions", "sum"),
    engagement_sessions=("ga4_engaged_sessions", "sum"),
    total_sessions=("ga4_sessions", "sum"),
    organic_sessions=("sessions_organic", "sum"),
).reset_index()

features["ctr"] = features["ctr"] / features["gsc_impressions_feat"].replace(0, 1)
features["engagement_rate"] = features["engagement_sessions"] / features["total_sessions"].replace(0, 1)
features["organic_share"] = features["organic_sessions"] / features["total_sessions"].replace(0, 1)

feature_frame = features[["content_hash_id", "ctr", "engagement_rate", "organic_share",
                            "gsc_impressions_feat", "total_sessions"]]

In [14]:
final = feature_frame.merge(outcome[["content_hash_id", "refresh_needed"]], on="content_hash_id")

In [15]:
print(final.columns)

Index(['content_hash_id', 'ctr', 'engagement_rate', 'organic_share',
       'gsc_impressions_feat', 'total_sessions', 'refresh_needed'],
      dtype='object')


In [16]:
final_1 = final.drop(columns=["content_hash_id"])

In [17]:
print(final_1.columns)

Index(['ctr', 'engagement_rate', 'organic_share', 'gsc_impressions_feat',
       'total_sessions', 'refresh_needed'],
      dtype='object')


In [18]:
X = final_1.drop(columns=['refresh_needed'])
y = final_1['refresh_needed']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
honest_score = f1_score(y_test, model.predict(X_test))
print(f"Calculated baseline F1: {honest_score:.3f}")

Calculated baseline F1: 0.285


**Add leakage feature:** gsc_impressions_late

In [20]:
final_2 = final.merge(outcome[["content_hash_id", "gsc_impressions_late"]], on="content_hash_id")

In [21]:
print(final_2.columns)

Index(['content_hash_id', 'ctr', 'engagement_rate', 'organic_share',
       'gsc_impressions_feat', 'total_sessions', 'refresh_needed',
       'gsc_impressions_late'],
      dtype='object')


In [22]:
X = final_2.drop(columns=['content_hash_id','refresh_needed'])
y = final_2['refresh_needed']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
honest_score = f1_score(y_test, model.predict(X_test))
print(f"Calculated baseline F1 after leakage feature add: {honest_score:.3f}")

Calculated baseline F1 after leakage feature add: 0.438


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**NOTE:** These are just measured and observed, not depecting actual meaning.

**Unbalanced history**: <br> per-page date coverage is heavily skewed: mean of ~2.84 report_dates per content_hash_id, with 858 pages having only a single date and a long tail (up to 10 dates) for the rest. This means the early-vs-late comparison isn't equally reliable across pages — a page with many daily rows in the early half gives a stable signal, while a page with just one or two points can't show a real trend, only a snapshot. The data can never distinguish genuine decline from noise for these sparsely-tracked pages.<br>
**GSC-only early rows**:<br> client_has_gsc/client_has_ga4 and gsc_data_available/ga4_data_available confirm that not every row has both search-visibility (GSC) and on-page behavior (GA4) data. Some rows only have one source successfully pulled, meaning the data can't always tell you the full performance picture — for those rows, it's a structural gap, not a genuine zero.<br>
**Window overlaps**:<br> Because the early/late split for this exercise was drawn from a single month (with the full pipeline drawn from a 90-day slice of 18 months of history), report dates near the split boundary can land somewhat arbitrarily in either window, since dates are irregular and sparse per page. This means the early/late boundary introduces some unavoidable ambiguity — a page's real behavior might not have meaningfully changed right at that cutoff, but the data has no way to reflect that nuance.

In [24]:
date_counts = unit_analysis_data.groupby("content_hash_id")["report_date"].nunique()

print("=== Unbalanced history ===")
print(date_counts.describe())
print("\nDistribution of dates-per-page:")
print(date_counts.value_counts().sort_index())
print(f"\nPages with only 1 date (no trend possible at all): {(date_counts == 1).sum()}")
print(f"Pages with only 2 dates (minimal, noisy trend): {(date_counts == 2).sum()}")
print(f"Total pages: {len(date_counts)}")
print(f"% of pages with <=2 dates: {(date_counts <= 2).mean():.1%}")

=== Claim 1: Unbalanced history ===
count    362172.000000
mean         28.783920
std           4.636891
min           1.000000
25%          30.000000
50%          30.000000
75%          30.000000
max          30.000000
Name: report_date, dtype: float64

Distribution of dates-per-page:
report_date
1        858
2        671
3        686
4        760
5       1520
6        924
7        930
8       1517
9        923
10       913
11      1195
12       976
13       715
14       908
15      1015
16      1463
17      1552
18       213
19       461
20       153
21      1803
22      3774
23       615
24       102
25      1405
26      3361
27       248
28       327
29       696
30    331488
Name: count, dtype: int64

Pages with only 1 date (no trend possible at all): 858
Pages with only 2 dates (minimal, noisy trend): 671
Total pages: 362172
% of pages with <=2 dates: 0.4%


In [ ]:
print("\n=== GSC-only / partial data availability ===")

total_rows = len(unit_analysis_data)
gsc_only = unit_analysis_data[(unit_analysis_data["gsc_data_available"] == True) & (unit_analysis_data["ga4_data_available"] == False)]
ga4_only = unit_analysis_data[(unit_analysis_data["gsc_data_available"] == False) & (unit_analysis_data["ga4_data_available"] == True)]
both_available = unit_analysis_data[(unit_analysis_data["gsc_data_available"] == True) & (unit_analysis_data["ga4_data_available"] == True)]
neither_available = unit_analysis_data[(unit_analysis_data["gsc_data_available"] == False) & (unit_analysis_data["ga4_data_available"] == False)]

print(f"Total rows: {total_rows}")
print(f"Rows with BOTH GSC and GA4 available: {len(both_available)} ({len(both_available)/total_rows:.1%})")
print(f"Rows with GSC-only (GA4 missing):     {len(gsc_only)} ({len(gsc_only)/total_rows:.1%})")
print(f"Rows with GA4-only (GSC missing):     {len(ga4_only)} ({len(ga4_only)/total_rows:.1%})")
print(f"Rows with NEITHER available:          {len(neither_available)} ({len(neither_available)/total_rows:.1%})")

# does this partial-availability issue skew toward earlier dates specifically?
data_sorted = unit_analysis_data.sort_values("report_date")
early_half = data_sorted.iloc[: len(data_sorted) // 2]
late_half = data_sorted.iloc[len(data_sorted) // 2 :]
print(f"\nGA4 availability rate, EARLY half of data: {early_half['ga4_data_available'].mean():.1%}")
print(f"GA4 availability rate, LATE half of data:  {late_half['ga4_data_available'].mean():.1%}")
print("(A meaningfully lower early-half rate would confirm 'GSC-only early rows'.)")


=== Claim 2: GSC-only / partial data availability ===
Total rows: 10424730
Rows with BOTH GSC and GA4 available: 462315 (4.4%)
Rows with GSC-only (GA4 missing):     2399336 (23.0%)
Rows with GA4-only (GSC missing):     61976 (0.6%)
Rows with NEITHER available:          5285052 (50.7%)


In [4]:
print("\n===  Window overlaps / boundary ambiguity ===")

month_df = unit_analysis_data.copy()
month_df["report_date"] = pd.to_datetime(month_df["report_date"])

mid_point = month_df["report_date"].min() + (month_df["report_date"].max() - month_df["report_date"].min()) / 2
window_days = 3  # +/- 3 days around the cutoff, adjust as needed

near_boundary = month_df[
    (month_df["report_date"] >= mid_point - pd.Timedelta(days=window_days)) &
    (month_df["report_date"] <= mid_point + pd.Timedelta(days=window_days))
]

print(f"Split point (mid-month): {mid_point.date()}")
print(f"Total rows this month: {len(month_df)}")
print(f"Rows within +/-{window_days} days of the split boundary: {len(near_boundary)} "
      f"({len(near_boundary)/len(month_df):.1%})")
print(f"Unique pages with data near the boundary: {near_boundary['content_hash_id'].nunique()}")
print("\n(These rows could shift from 'early' to 'late' -- or vice versa -- with a")
print(" slightly different cutoff choice, even though the underlying page behavior")
print(" didn't meaningfully change right at that date. This is the window-overlap")
print(" ambiguity: the label near the boundary is sensitive to an essentially")
print(" arbitrary cutoff decision, not a clean behavioral shift.)")


=== Claim 3: Window overlaps / boundary ambiguity ===
Split point (mid-month): 2026-04-15
Total rows this month: 10424730
Rows within +/-3 days of the split boundary: 2087064 (20.0%)
Unique pages with data near the boundary: 350299

(These rows could shift from 'early' to 'late' -- or vice versa -- with a
 slightly different cutoff choice, even though the underlying page behavior
 didn't meaningfully change right at that date. This is the window-overlap
 ambiguity: the label near the boundary is sensitive to an essentially
 arbitrary cutoff decision, not a clean behavioral shift.)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.